
# Qwen3 4B ASR Retrieval Server — Kaggle 2×T4

**Mục tiêu:** host model embedding làm server để web/backend của `Multimodal-Retrieval` có thể search semantic trên ASR vectors trong Milvus/Zilliz.

## Quan trọng: dùng Qwen3-Embedding-4B, không dùng Qwen3-VL-4B-Instruct

Trong repo hiện tại, notebook `fe-asr-text-embedding-v1.ipynb` tạo ASR vectors bằng:

- `Qwen/Qwen3-Embedding-4B`
- dimension `2560`
- `normalize_embeddings=True`
- `max_seq_length=2048`
- batch size offline hiện tại: `8`

Vì query vector và ASR document vector phải nằm **cùng vector space**, server này host `Qwen/Qwen3-Embedding-4B`. `Qwen3-VL-4B-Instruct` là VLM sinh nội dung, không phải checkpoint tương ứng với ASR embedding đã tạo; dùng nó để query Milvus collection 2560-dim này sẽ sai kiến trúc retrieval.

Tên notebook vẫn giữ đúng yêu cầu: `qwen3_4b_server_v1.ipynb`.

## Kiến trúc runtime

```text
React web
   |
   v
FastAPI backend hiện tại
   |
   | OpenAI-compatible /v1/embeddings
   | hoặc /v1/asr/search
   v
Kaggle server
   |
   +--> async bounded queue
   +--> dynamic micro-batcher
           |                 |
           v                 v
      GPU worker 0      GPU worker 1
      T4 replica        T4 replica
      Qwen3-4B FP16     Qwen3-4B FP16
           \                 /
            \               /
             +---- query vector (2560, L2 normalized)
                         |
                         v
                    Milvus/Zilliz
                    COSINE search
```

### Tại sao không chạy `uvicorn --workers 2`?

Mỗi Uvicorn worker là một process riêng. Nếu mỗi worker tự load model cho cả hai GPU thì model bị nhân bản ngoài kiểm soát và rất dễ tràn VRAM. Notebook này dùng **1 HTTP process**, bên trong spawn **1 model process/GPU**, nên 2×T4 trở thành 2 inference replicas rõ ràng.

### Endpoint

- `GET /health`
- `GET /v1/models`
- `POST /v1/embeddings` — OpenAI-compatible, dùng trực tiếp với adapter hiện có của repo
- `POST /v1/asr/search` — embed query rồi search Milvus trực tiếp
- `GET /metrics`

Nguồn tham chiếu kỹ thuật chính:

- Qwen3-Embedding-4B model card: `https://huggingface.co/Qwen/Qwen3-Embedding-4B`
- Qwen3 Embedding blog: `https://qwenlm.github.io/blog/qwen3-embedding/`
- NVIDIA T4 specs: `https://www.nvidia.com/en-us/data-center/tesla-t4/`


In [ ]:

# 1) Install runtime dependencies.
# Do NOT install flash-attn on T4 here; this server uses PyTorch SDPA for portability.

%pip install -q -U \
  "transformers>=4.51,<6" \
  "sentence-transformers>=5.1,<6" \
  "fastapi>=0.115" \
  "uvicorn[standard]>=0.30" \
  "orjson>=3.10" \
  "pymilvus>=2.5" \
  "httpx>=0.27" \
  "pyngrok>=7.2" \
  "psutil>=6.0"


In [ ]:

# 2) Inspect Kaggle hardware and create runtime configuration.

import os
import secrets
import subprocess
from pathlib import Path

print("CPU count:", os.cpu_count())
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

# ------------------------------------------------------------------
# Model
# ------------------------------------------------------------------
# Online mode:
os.environ.setdefault("QWEN_MODEL_PATH", "Qwen/Qwen3-Embedding-4B")

# Offline Kaggle mode:
# Attach Qwen3-Embedding-4B as a Kaggle Model/Dataset, then replace with:
# os.environ["QWEN_MODEL_PATH"] = "/kaggle/input/.../qwen3-embedding-4b/.../1"

os.environ["MODEL_ID"] = "Qwen/Qwen3-Embedding-4B"
os.environ["EMBEDDING_DIM"] = "2560"

# Query traffic is usually much shorter than ASR documents.
os.environ.setdefault("MAX_SEQ_LENGTH", "512")

# Conservative stable defaults for T4. Benchmark below and tune if VRAM allows.
os.environ.setdefault("INFER_BATCH_SIZE", "8")
os.environ.setdefault("MAX_BATCH_TEXTS", "32")
os.environ.setdefault("BATCH_WAIT_MS", "8")
os.environ.setdefault("QUEUE_MAX_REQUESTS", "1024")
os.environ.setdefault("REQUEST_TIMEOUT_S", "45")

# Automatically use all visible GPUs: on Kaggle T4x2 this becomes 2 workers.
try:
    gpu_lines = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"],
        text=True,
    ).strip().splitlines()
    gpu_count = len([x for x in gpu_lines if x.strip()])
except Exception:
    gpu_count = 0

os.environ["GPU_WORKERS"] = str(max(1, gpu_count))
cpu_count = os.cpu_count() or 4
os.environ["CPU_THREADS_PER_GPU"] = str(max(1, cpu_count // max(1, gpu_count or 1)))

# ------------------------------------------------------------------
# Security
# ------------------------------------------------------------------
# Generated once for this notebook session. Put this token only in your backend,
# not directly in React source code.
os.environ.setdefault("SERVER_API_KEY", secrets.token_urlsafe(32))

# Set your frontend origin only if the browser must call this server directly.
# Preferred flow: browser -> existing backend -> this server.
# os.environ["CORS_ORIGINS"] = "https://your-frontend.example.com"

# ------------------------------------------------------------------
# Milvus/Zilliz
# ------------------------------------------------------------------
# Fill these before launching if you want /v1/asr/search.
# If you only want /v1/embeddings, MILVUS_URI may stay empty.
#
# os.environ["MILVUS_URI"] = "https://<cluster-endpoint>"
# os.environ["MILVUS_TOKEN"] = "<user:password or API token>"
os.environ.setdefault(
    "MILVUS_COLLECTION",
    "asr_embeddings_qwen3_embedding_4b_v1",
)
os.environ.setdefault("MILVUS_CLIENT_POOL_SIZE", "4")

print("GPU workers:", os.environ["GPU_WORKERS"])
print("CPU threads / GPU worker:", os.environ["CPU_THREADS_PER_GPU"])
print("Model:", os.environ["QWEN_MODEL_PATH"])
print("API key:", os.environ["SERVER_API_KEY"])
print("Milvus collection:", os.environ["MILVUS_COLLECTION"])


In [ ]:
from pathlib import Path

SERVER_CODE = r'''
from __future__ import annotations

import asyncio
import json
import math
import multiprocessing as mp
import os
import queue as thread_queue
import secrets
import subprocess
import threading
import time
import traceback
from contextlib import asynccontextmanager
from dataclasses import dataclass
from typing import Any, Literal

import numpy as np
from fastapi import Depends, FastAPI, Header, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import ORJSONResponse
from pydantic import BaseModel, Field


# ---------------------------------------------------------------------
# Runtime configuration
# ---------------------------------------------------------------------

MODEL_ID = os.getenv("MODEL_ID", "Qwen/Qwen3-Embedding-4B")
MODEL_PATH = os.getenv("QWEN_MODEL_PATH", MODEL_ID)
MODEL_CACHE = os.getenv("MODEL_CACHE", "/kaggle/working/hf-cache")
EMBEDDING_DIM = int(os.getenv("EMBEDDING_DIM", "2560"))

# ASR passages in the repository were embedded with max_seq_length=2048.
# Online user queries are normally short; 512 lowers activation memory and
# improves batching. Raise this only if your search queries are genuinely long.
MAX_SEQ_LENGTH = int(os.getenv("MAX_SEQ_LENGTH", "512"))

# Model.encode internal batch size per GPU. For T4, 8 is a conservative default.
# Dynamic batching can merge more request texts, while SentenceTransformer
# still slices the merged batch into INFER_BATCH_SIZE chunks.
INFER_BATCH_SIZE = int(os.getenv("INFER_BATCH_SIZE", "8"))

# Dynamic batching across HTTP requests.
MAX_BATCH_TEXTS = int(os.getenv("MAX_BATCH_TEXTS", "32"))
BATCH_WAIT_MS = float(os.getenv("BATCH_WAIT_MS", "8"))
QUEUE_MAX_REQUESTS = int(os.getenv("QUEUE_MAX_REQUESTS", "1024"))
MAX_TEXTS_PER_REQUEST = int(os.getenv("MAX_TEXTS_PER_REQUEST", "64"))
MAX_INPUT_CHARS = int(os.getenv("MAX_INPUT_CHARS", "16000"))
REQUEST_TIMEOUT_S = float(os.getenv("REQUEST_TIMEOUT_S", "45"))

# Public-server protection. Leave empty only for private/local testing.
SERVER_API_KEY = os.getenv("SERVER_API_KEY", "").strip()

# Qwen recommends task-specific English instructions for multilingual retrieval.
QUERY_INSTRUCTION = os.getenv(
    "QUERY_INSTRUCTION",
    "Given a user query for video retrieval, retrieve spoken ASR transcript "
    "segments from videos that are semantically relevant to the query",
)

# Milvus/Zilliz settings. Direct /v1/asr/search is enabled only when URI is set.
MILVUS_URI = os.getenv("MILVUS_URI", "").strip()
MILVUS_TOKEN = os.getenv("MILVUS_TOKEN", "").strip()
MILVUS_COLLECTION = os.getenv(
    "MILVUS_COLLECTION",
    "asr_embeddings_qwen3_embedding_4b_v1",
)
MILVUS_SEARCH_TIMEOUT_S = float(os.getenv("MILVUS_SEARCH_TIMEOUT_S", "12"))
MILVUS_CLIENT_POOL_SIZE = int(os.getenv("MILVUS_CLIENT_POOL_SIZE", "4"))

# CORS should usually point to your frontend origin. Browser -> existing backend
# -> this service is preferred because it keeps SERVER_API_KEY off the browser.
_raw_origins = os.getenv("CORS_ORIGINS", "").strip()
CORS_ORIGINS = [x.strip() for x in _raw_origins.split(",") if x.strip()]


def detect_gpu_count() -> int:
    try:
        raw = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"],
            text=True,
            stderr=subprocess.DEVNULL,
        )
        return max(0, len([line for line in raw.splitlines() if line.strip()]))
    except Exception:
        return 0


GPU_WORKERS = int(os.getenv("GPU_WORKERS", str(detect_gpu_count())))
if GPU_WORKERS <= 0:
    GPU_WORKERS = 1

CPU_COUNT = os.cpu_count() or 4
CPU_THREADS_PER_GPU = int(
    os.getenv("CPU_THREADS_PER_GPU", str(max(1, CPU_COUNT // max(1, GPU_WORKERS))))
)


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def _format_text(text: str, input_type: str) -> str:
    text = text.strip()
    if input_type == "query":
        return f"Instruct: {QUERY_INSTRUCTION}\nQuery:{text}"
    return text


def _gpu_worker_main(gpu_id: int, conn, cfg: dict[str, Any]) -> None:
    """
    One process owns exactly one GPU and one model replica.

    This avoids uvicorn worker duplication and makes 2xT4 operate as two
    independent embedding replicas for throughput.
    """
    try:
        # Keep tokenizer/BLAS CPU pools bounded per GPU process.
        os.environ["TOKENIZERS_PARALLELISM"] = "true"
        os.environ["RAYON_NUM_THREADS"] = str(cfg["cpu_threads_per_gpu"])
        os.environ["OMP_NUM_THREADS"] = str(cfg["cpu_threads_per_gpu"])
        os.environ["MKL_NUM_THREADS"] = str(cfg["cpu_threads_per_gpu"])

        import torch
        from sentence_transformers import SentenceTransformer

        if not torch.cuda.is_available():
            raise RuntimeError("CUDA is not available. Enable Kaggle GPU accelerator.")

        available = torch.cuda.device_count()
        if gpu_id >= available:
            raise RuntimeError(
                f"Requested cuda:{gpu_id}, but this process sees only {available} CUDA device(s)."
            )

        device = f"cuda:{gpu_id}"
        torch.cuda.set_device(gpu_id)
        torch.set_num_threads(max(1, int(cfg["cpu_threads_per_gpu"])))
        try:
            torch.set_num_interop_threads(1)
        except RuntimeError:
            pass

        # T4 is Turing: use FP16. FlashAttention2 is intentionally not required.
        # SDPA is portable and lets PyTorch choose an available attention kernel.
        model = SentenceTransformer(
            cfg["model_path"],
            device=device,
            cache_folder=cfg["model_cache"],
            trust_remote_code=True,
            model_kwargs={
                "torch_dtype": torch.float16,
                "attn_implementation": "sdpa",
            },
            tokenizer_kwargs={"padding_side": "left"},
        )
        model.max_seq_length = int(cfg["max_seq_length"])
        model.eval()

        dim = int(model.get_sentence_embedding_dimension() or 0)
        if dim and dim != int(cfg["embedding_dim"]):
            raise RuntimeError(
                f"Embedding dimension mismatch: model={dim}, expected={cfg['embedding_dim']}"
            )

        def encode_adaptive(texts: list[str]) -> np.ndarray:
            if not texts:
                return np.empty((0, int(cfg["embedding_dim"])), dtype=np.float32)

            try:
                with torch.inference_mode():
                    result = model.encode(
                        texts,
                        batch_size=min(int(cfg["infer_batch_size"]), len(texts)),
                        show_progress_bar=False,
                        convert_to_numpy=True,
                        normalize_embeddings=True,
                    )
                return np.asarray(result, dtype=np.float32)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if len(texts) <= 1:
                    raise
                mid = len(texts) // 2
                left = encode_adaptive(texts[:mid])
                right = encode_adaptive(texts[mid:])
                return np.concatenate([left, right], axis=0)

        # Warmup to allocate kernels/caches before accepting traffic.
        warmup = _format_text("video có người đang nói chuyện", "query")
        torch.cuda.synchronize(gpu_id)
        _ = encode_adaptive([warmup])
        torch.cuda.synchronize(gpu_id)

        props = torch.cuda.get_device_properties(gpu_id)
        conn.send(
            {
                "type": "ready",
                "ok": True,
                "gpu_id": gpu_id,
                "gpu_name": props.name,
                "total_memory_gb": round(props.total_memory / (1024**3), 2),
                "embedding_dim": int(cfg["embedding_dim"]),
                "max_seq_length": int(cfg["max_seq_length"]),
            }
        )

        while True:
            msg = conn.recv()
            cmd = msg.get("cmd")

            if cmd == "shutdown":
                conn.send({"ok": True, "cmd": "shutdown"})
                break

            if cmd != "embed":
                conn.send({"ok": False, "error": f"Unknown command: {cmd}"})
                continue

            texts = list(msg.get("texts") or [])
            input_types = list(msg.get("input_types") or [])
            if len(texts) != len(input_types):
                conn.send({"ok": False, "error": "texts/input_types length mismatch"})
                continue

            formatted = [_format_text(t, typ) for t, typ in zip(texts, input_types)]
            started = time.perf_counter()
            torch.cuda.synchronize(gpu_id)
            vectors = encode_adaptive(formatted)
            torch.cuda.synchronize(gpu_id)
            latency_ms = (time.perf_counter() - started) * 1000.0

            conn.send(
                {
                    "ok": True,
                    "vectors": vectors,
                    "latency_ms": latency_ms,
                    "count": len(formatted),
                    "gpu_id": gpu_id,
                }
            )

    except EOFError:
        pass
    except Exception:
        try:
            conn.send(
                {
                    "type": "ready",
                    "ok": False,
                    "gpu_id": gpu_id,
                    "error": traceback.format_exc(),
                }
            )
        except Exception:
            pass
    finally:
        try:
            conn.close()
        except Exception:
            pass


class GPUProcess:
    def __init__(self, gpu_id: int, cfg: dict[str, Any]) -> None:
        self.gpu_id = gpu_id
        self.cfg = cfg
        self._lock = threading.Lock()
        ctx = mp.get_context("spawn")
        self._parent_conn, child_conn = ctx.Pipe(duplex=True)
        self._process = ctx.Process(
            target=_gpu_worker_main,
            args=(gpu_id, child_conn, cfg),
            daemon=True,
            name=f"qwen3-embedding-gpu-{gpu_id}",
        )
        self._process.start()
        child_conn.close()
        self.ready_info: dict[str, Any] | None = None

    def wait_ready(self, timeout_s: float = 1200.0) -> dict[str, Any]:
        if not self._parent_conn.poll(timeout_s):
            raise TimeoutError(f"GPU worker {self.gpu_id} did not become ready.")
        msg = self._parent_conn.recv()
        if not msg.get("ok"):
            raise RuntimeError(
                f"GPU worker {self.gpu_id} failed to start:\n{msg.get('error', msg)}"
            )
        self.ready_info = msg
        return msg

    def embed(self, texts: list[str], input_types: list[str]) -> dict[str, Any]:
        if not self.is_alive():
            raise RuntimeError(f"GPU worker {self.gpu_id} is not alive.")
        with self._lock:
            self._parent_conn.send(
                {
                    "cmd": "embed",
                    "texts": texts,
                    "input_types": input_types,
                }
            )
            msg = self._parent_conn.recv()
        if not msg.get("ok"):
            raise RuntimeError(msg.get("error", "GPU worker failed."))
        return msg

    def is_alive(self) -> bool:
        return self._process.is_alive()

    def close(self) -> None:
        if self._process.is_alive():
            try:
                with self._lock:
                    self._parent_conn.send({"cmd": "shutdown"})
                    if self._parent_conn.poll(5):
                        self._parent_conn.recv()
            except Exception:
                pass
            self._process.join(timeout=5)
        if self._process.is_alive():
            self._process.terminate()
            self._process.join(timeout=5)
        try:
            self._parent_conn.close()
        except Exception:
            pass


@dataclass(slots=True)
class PendingJob:
    texts: list[str]
    input_type: str
    future: asyncio.Future


class QueueOverloaded(RuntimeError):
    pass


class EmbeddingScheduler:
    """
    Async request queue + one batch loop per GPU process.

    Each loop merges independent HTTP requests for BATCH_WAIT_MS until
    MAX_BATCH_TEXTS is reached, then sends one inference command to its GPU.
    """

    def __init__(self, workers: list[GPUProcess]) -> None:
        self.workers = workers
        self.queue: asyncio.Queue[PendingJob] = asyncio.Queue(
            maxsize=QUEUE_MAX_REQUESTS
        )
        self.tasks: list[asyncio.Task] = []
        self.total_requests = 0
        self.total_texts = 0
        self.total_batches = 0
        self.total_gpu_ms = 0.0
        self.last_batch_size_by_gpu: dict[int, int] = {}
        self.last_gpu_ms_by_gpu: dict[int, float] = {}

    async def start(self) -> None:
        for worker in self.workers:
            self.tasks.append(
                asyncio.create_task(
                    self._batch_loop(worker),
                    name=f"batch-loop-gpu-{worker.gpu_id}",
                )
            )

    async def stop(self) -> None:
        for task in self.tasks:
            task.cancel()
        await asyncio.gather(*self.tasks, return_exceptions=True)
        self.tasks.clear()

    async def embed(self, texts: list[str], input_type: str) -> np.ndarray:
        loop = asyncio.get_running_loop()
        future = loop.create_future()
        job = PendingJob(texts=texts, input_type=input_type, future=future)

        try:
            self.queue.put_nowait(job)
        except asyncio.QueueFull as exc:
            raise QueueOverloaded(
                f"Embedding queue is full ({QUEUE_MAX_REQUESTS} pending requests)."
            ) from exc

        self.total_requests += 1
        self.total_texts += len(texts)

        try:
            return await asyncio.wait_for(
                asyncio.shield(future),
                timeout=REQUEST_TIMEOUT_S,
            )
        except asyncio.TimeoutError as exc:
            raise TimeoutError(
                f"Embedding request exceeded {REQUEST_TIMEOUT_S}s."
            ) from exc

    async def _batch_loop(self, worker: GPUProcess) -> None:
        loop = asyncio.get_running_loop()

        while True:
            first = await self.queue.get()
            jobs = [first]
            total_texts = len(first.texts)
            deadline = loop.time() + (BATCH_WAIT_MS / 1000.0)

            while total_texts < MAX_BATCH_TEXTS:
                remaining = deadline - loop.time()
                if remaining <= 0:
                    break

                try:
                    nxt = await asyncio.wait_for(self.queue.get(), timeout=remaining)
                except asyncio.TimeoutError:
                    break

                if total_texts + len(nxt.texts) > MAX_BATCH_TEXTS:
                    # Requeue it for another GPU/batch. Ordering is not semantically
                    # important for embedding requests.
                    self.queue.put_nowait(nxt)
                    break

                jobs.append(nxt)
                total_texts += len(nxt.texts)

            flat_texts: list[str] = []
            flat_types: list[str] = []
            spans: list[tuple[int, int, PendingJob]] = []
            cursor = 0

            for job in jobs:
                start = cursor
                flat_texts.extend(job.texts)
                flat_types.extend([job.input_type] * len(job.texts))
                cursor += len(job.texts)
                spans.append((start, cursor, job))

            try:
                msg = await asyncio.to_thread(
                    worker.embed,
                    flat_texts,
                    flat_types,
                )
                vectors = np.asarray(msg["vectors"], dtype=np.float32)

                if vectors.shape != (len(flat_texts), EMBEDDING_DIM):
                    raise RuntimeError(
                        f"Invalid worker output shape {vectors.shape}; "
                        f"expected ({len(flat_texts)}, {EMBEDDING_DIM})."
                    )

                gpu_ms = float(msg.get("latency_ms", 0.0))
                self.total_batches += 1
                self.total_gpu_ms += gpu_ms
                self.last_batch_size_by_gpu[worker.gpu_id] = len(flat_texts)
                self.last_gpu_ms_by_gpu[worker.gpu_id] = gpu_ms

                for start, end, job in spans:
                    if not job.future.done():
                        job.future.set_result(vectors[start:end])

            except Exception as exc:
                for _, _, job in spans:
                    if not job.future.done():
                        job.future.set_exception(exc)
            finally:
                for _ in jobs:
                    self.queue.task_done()

    def stats(self) -> dict[str, Any]:
        return {
            "queue_size": self.queue.qsize(),
            "queue_capacity": QUEUE_MAX_REQUESTS,
            "total_requests": self.total_requests,
            "total_texts": self.total_texts,
            "total_batches": self.total_batches,
            "avg_gpu_batch_ms": (
                round(self.total_gpu_ms / self.total_batches, 3)
                if self.total_batches
                else 0.0
            ),
            "last_batch_size_by_gpu": self.last_batch_size_by_gpu,
            "last_gpu_ms_by_gpu": {
                k: round(v, 3) for k, v in self.last_gpu_ms_by_gpu.items()
            },
        }


# ---------------------------------------------------------------------
# Milvus pool
# ---------------------------------------------------------------------

class MilvusPool:
    def __init__(self) -> None:
        self._pool: thread_queue.LifoQueue = thread_queue.LifoQueue(
            maxsize=max(1, MILVUS_CLIENT_POOL_SIZE)
        )
        self._initialized = False
        self._init_lock = threading.Lock()

    @property
    def enabled(self) -> bool:
        return bool(MILVUS_URI)

    def _ensure(self) -> None:
        if self._initialized:
            return
        with self._init_lock:
            if self._initialized:
                return
            if not self.enabled:
                raise RuntimeError("MILVUS_URI is not configured.")

            from pymilvus import MilvusClient

            for _ in range(max(1, MILVUS_CLIENT_POOL_SIZE)):
                kwargs: dict[str, Any] = {"uri": MILVUS_URI}
                if MILVUS_TOKEN:
                    kwargs["token"] = MILVUS_TOKEN
                self._pool.put(MilvusClient(**kwargs))
            self._initialized = True

    @staticmethod
    def _quote(value: str) -> str:
        return value.replace("\\", "\\\\").replace('"', '\\"')

    def search(
        self,
        vector: list[float],
        top_k: int,
        video_id: str | None,
        min_score: float | None,
    ) -> list[dict[str, Any]]:
        self._ensure()
        client = self._pool.get()
        try:
            filters = ['doc_type == "asr"']
            if video_id:
                filters.append(f'video_id == "{self._quote(video_id)}"')
            filter_expr = " and ".join(filters)

            raw = client.search(
                collection_name=MILVUS_COLLECTION,
                data=[vector],
                anns_field="vector",
                limit=top_k,
                filter=filter_expr,
                output_fields=[
                    "keyframe_id",
                    "video_id",
                    "text",
                    "doc_type",
                    "model_version",
                    "metadata_json",
                ],
                search_params={"metric_type": "COSINE", "params": {}},
                timeout=MILVUS_SEARCH_TIMEOUT_S,
            )

            result: list[dict[str, Any]] = []
            for hit in raw[0] if raw else []:
                score = float(hit.get("distance", 0.0))
                if min_score is not None and score < min_score:
                    continue

                entity = hit.get("entity", {}) or {}
                metadata: dict[str, Any] = {}
                raw_meta = entity.get("metadata_json")
                if raw_meta:
                    try:
                        metadata = json.loads(raw_meta)
                    except Exception:
                        metadata = {"raw_metadata_json": raw_meta}

                result.append(
                    {
                        "id": str(hit.get("id")),
                        "score": score,
                        "keyframe_id": entity.get("keyframe_id"),
                        "video_id": entity.get("video_id"),
                        "text": entity.get("text"),
                        "doc_type": entity.get("doc_type"),
                        "model_version": entity.get("model_version"),
                        "metadata": metadata,
                    }
                )
            return result
        finally:
            self._pool.put(client)


# ---------------------------------------------------------------------
# FastAPI schemas
# ---------------------------------------------------------------------

class EmbeddingRequest(BaseModel):
    model: str = MODEL_ID
    input: str | list[str]
    input_type: Literal["query", "document"] = "query"
    dimensions: int | None = None


class ASRSearchRequest(BaseModel):
    query: str = Field(min_length=1)
    top_k: int = Field(default=50, ge=1, le=200)
    video_id: str | None = None
    min_score: float | None = Field(default=None, ge=-1.0, le=1.0)
    return_query_vector: bool = False


# ---------------------------------------------------------------------
# FastAPI application
# ---------------------------------------------------------------------

scheduler: EmbeddingScheduler | None = None
gpu_workers: list[GPUProcess] = []
milvus_pool = MilvusPool()


def _worker_cfg() -> dict[str, Any]:
    return {
        "model_path": MODEL_PATH,
        "model_cache": MODEL_CACHE,
        "embedding_dim": EMBEDDING_DIM,
        "max_seq_length": MAX_SEQ_LENGTH,
        "infer_batch_size": INFER_BATCH_SIZE,
        "cpu_threads_per_gpu": CPU_THREADS_PER_GPU,
    }


@asynccontextmanager
async def lifespan(app: FastAPI):
    global scheduler, gpu_workers

    available = detect_gpu_count()
    if available <= 0:
        raise RuntimeError(
            "No NVIDIA GPU detected. In Kaggle, enable a GPU accelerator first."
        )

    worker_count = min(GPU_WORKERS, available)
    gpu_workers = [GPUProcess(i, _worker_cfg()) for i in range(worker_count)]

    ready = await asyncio.gather(
        *[asyncio.to_thread(worker.wait_ready) for worker in gpu_workers]
    )

    for info in ready:
        if int(info.get("embedding_dim", 0)) != EMBEDDING_DIM:
            raise RuntimeError(
                f"Worker dimension mismatch: {info.get('embedding_dim')} != {EMBEDDING_DIM}"
            )

    scheduler = EmbeddingScheduler(gpu_workers)
    await scheduler.start()

    yield

    if scheduler is not None:
        await scheduler.stop()
    for worker in gpu_workers:
        worker.close()


app = FastAPI(
    title="Qwen3-Embedding-4B ASR Retrieval Server",
    version="1.0.0",
    default_response_class=ORJSONResponse,
    lifespan=lifespan,
)

if CORS_ORIGINS:
    app.add_middleware(
        CORSMiddleware,
        allow_origins=CORS_ORIGINS,
        allow_credentials=False,
        allow_methods=["GET", "POST"],
        allow_headers=["Authorization", "Content-Type", "X-API-Key"],
    )


def require_auth(
    authorization: str | None = Header(default=None),
    x_api_key: str | None = Header(default=None),
) -> None:
    if not SERVER_API_KEY:
        return

    provided = x_api_key or ""
    if authorization and authorization.lower().startswith("bearer "):
        provided = authorization[7:].strip()

    if not provided or not secrets.compare_digest(provided, SERVER_API_KEY):
        raise HTTPException(status_code=401, detail="Invalid API key.")


def _validate_texts(value: str | list[str]) -> list[str]:
    texts = [value] if isinstance(value, str) else list(value)
    if not texts:
        raise HTTPException(status_code=400, detail="input must not be empty.")
    if len(texts) > MAX_TEXTS_PER_REQUEST:
        raise HTTPException(
            status_code=413,
            detail=f"Too many texts: {len(texts)} > {MAX_TEXTS_PER_REQUEST}.",
        )

    cleaned: list[str] = []
    for text in texts:
        if not isinstance(text, str):
            raise HTTPException(status_code=400, detail="Every input must be a string.")
        text = text.strip()
        if not text:
            raise HTTPException(status_code=400, detail="Input text must not be blank.")
        if len(text) > MAX_INPUT_CHARS:
            raise HTTPException(
                status_code=413,
                detail=f"Input exceeds MAX_INPUT_CHARS={MAX_INPUT_CHARS}.",
            )
        cleaned.append(text)
    return cleaned


@app.get("/health")
async def health() -> dict[str, Any]:
    workers = [
        {
            "gpu_id": w.gpu_id,
            "alive": w.is_alive(),
            "ready": w.ready_info,
        }
        for w in gpu_workers
    ]
    return {
        "status": (
            "ok"
            if scheduler is not None and all(w["alive"] for w in workers)
            else "degraded"
        ),
        "model": MODEL_ID,
        "model_path": MODEL_PATH,
        "embedding_dim": EMBEDDING_DIM,
        "gpu_workers": workers,
        "cpu_count": CPU_COUNT,
        "cpu_threads_per_gpu": CPU_THREADS_PER_GPU,
        "scheduler": scheduler.stats() if scheduler else None,
        "milvus_enabled": milvus_pool.enabled,
        "milvus_collection": MILVUS_COLLECTION if milvus_pool.enabled else None,
    }


@app.get("/v1/models")
async def list_models() -> dict[str, Any]:
    return {
        "object": "list",
        "data": [
            {
                "id": MODEL_ID,
                "object": "model",
                "owned_by": "Qwen",
            }
        ],
    }


@app.get("/metrics", dependencies=[Depends(require_auth)])
async def metrics() -> dict[str, Any]:
    return {
        "scheduler": scheduler.stats() if scheduler else None,
        "workers": [
            {
                "gpu_id": w.gpu_id,
                "alive": w.is_alive(),
                "ready": w.ready_info,
            }
            for w in gpu_workers
        ],
    }


@app.post("/v1/embeddings", dependencies=[Depends(require_auth)])
async def embeddings(req: EmbeddingRequest) -> dict[str, Any]:
    if req.model not in {MODEL_ID, "qwen3-embedding-4b", "Qwen3-Embedding-4B"}:
        raise HTTPException(
            status_code=400,
            detail=f"Unsupported model '{req.model}'. Expected '{MODEL_ID}'.",
        )

    if req.dimensions is not None and int(req.dimensions) != EMBEDDING_DIM:
        raise HTTPException(
            status_code=400,
            detail=(
                f"This service is locked to {EMBEDDING_DIM} dimensions because "
                "the ASR Milvus collection must use the same vector space."
            ),
        )

    texts = _validate_texts(req.input)
    if scheduler is None:
        raise HTTPException(status_code=503, detail="Embedding scheduler is not ready.")

    try:
        vectors = await scheduler.embed(texts, req.input_type)
    except QueueOverloaded as exc:
        raise HTTPException(status_code=429, detail=str(exc)) from exc
    except TimeoutError as exc:
        raise HTTPException(status_code=504, detail=str(exc)) from exc
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"Embedding failed: {exc}") from exc

    data = [
        {
            "object": "embedding",
            "index": i,
            "embedding": vectors[i].tolist(),
        }
        for i in range(len(texts))
    ]
    return {
        "object": "list",
        "model": MODEL_ID,
        "data": data,
    }


@app.post("/v1/asr/search", dependencies=[Depends(require_auth)])
async def asr_search(req: ASRSearchRequest) -> dict[str, Any]:
    if not milvus_pool.enabled:
        raise HTTPException(
            status_code=503,
            detail="MILVUS_URI is not configured; direct ASR search is disabled.",
        )
    if scheduler is None:
        raise HTTPException(status_code=503, detail="Embedding scheduler is not ready.")

    query = req.query.strip()
    if len(query) > MAX_INPUT_CHARS:
        raise HTTPException(status_code=413, detail="Query is too long.")

    try:
        vectors = await scheduler.embed([query], "query")
        query_vector = vectors[0].tolist()
    except QueueOverloaded as exc:
        raise HTTPException(status_code=429, detail=str(exc)) from exc
    except TimeoutError as exc:
        raise HTTPException(status_code=504, detail=str(exc)) from exc
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"Embedding failed: {exc}") from exc

    try:
        hits = await asyncio.to_thread(
            milvus_pool.search,
            query_vector,
            req.top_k,
            req.video_id,
            req.min_score,
        )
    except Exception as exc:
        raise HTTPException(status_code=502, detail=f"Milvus search failed: {exc}") from exc

    body: dict[str, Any] = {
        "model": MODEL_ID,
        "collection": MILVUS_COLLECTION,
        "query": query,
        "top_k": req.top_k,
        "count": len(hits),
        "hits": hits,
    }
    if req.return_query_vector:
        body["query_vector"] = query_vector
    return body
'''

Path('/kaggle/working/qwen3_server.py').write_text(SERVER_CODE, encoding='utf-8')
compile(SERVER_CODE, 'qwen3_server.py', 'exec')
print('Wrote /kaggle/working/qwen3_server.py')



## 4) Launch server

Server chạy **một Uvicorn process duy nhất**. Bên trong nó tự spawn `GPU_WORKERS` subprocesses, mỗi process giữ đúng một model replica trên một GPU.

Lần đầu nếu tải model từ Hugging Face, startup sẽ chỉ trả `/health` sau khi cả hai GPU workers đã load và warm up xong.


In [ ]:

# 4) Launch in the background and wait until /health is ready.

import os
import signal
import subprocess
import sys
import time
from pathlib import Path

PORT = int(os.environ.get("PORT", "8004"))
PID_FILE = Path("/kaggle/working/qwen3_server.pid")
LOG_FILE = Path("/kaggle/working/qwen3_server.log")

def stop_previous_server():
    if not PID_FILE.exists():
        return
    try:
        pid = int(PID_FILE.read_text().strip())
        os.kill(pid, signal.SIGTERM)
        time.sleep(2)
        try:
            os.kill(pid, 0)
            os.kill(pid, signal.SIGKILL)
        except OSError:
            pass
    except Exception:
        pass
    PID_FILE.unlink(missing_ok=True)

stop_previous_server()

log_handle = LOG_FILE.open("w", encoding="utf-8")
proc = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "qwen3_server:app",
        "--host",
        "0.0.0.0",
        "--port",
        str(PORT),
        "--workers",
        "1",
        "--log-level",
        "info",
    ],
    cwd="/kaggle/working",
    env=os.environ.copy(),
    stdout=log_handle,
    stderr=subprocess.STDOUT,
)

PID_FILE.write_text(str(proc.pid))
print("Server PID:", proc.pid)
print("Log:", LOG_FILE)

# Poll health. Model download/load can be the slowest part of first startup.
import httpx

health = None
for _ in range(600):
    if proc.poll() is not None:
        print(LOG_FILE.read_text(encoding="utf-8")[-12000:])
        raise RuntimeError(f"Server exited with code {proc.returncode}")
    try:
        r = httpx.get(f"http://127.0.0.1:{PORT}/health", timeout=2.0)
        if r.status_code == 200:
            health = r.json()
            break
    except Exception:
        pass
    time.sleep(2)

if health is None:
    print(LOG_FILE.read_text(encoding="utf-8")[-12000:])
    raise TimeoutError("Server did not become healthy.")

health


In [ ]:

# 5) OpenAI-compatible embedding smoke test.
# By default input_type="query", so the service applies the Qwen retrieval instruction.

import os
import httpx
import numpy as np

PORT = int(os.environ.get("PORT", "8004"))
API_KEY = os.environ["SERVER_API_KEY"]

payload = {
    "model": "Qwen/Qwen3-Embedding-4B",
    "input": [
        "một người đàn ông đang nói về biến đổi khí hậu",
        "phỏng vấn một cầu thủ bóng đá sau trận đấu",
    ],
    "input_type": "query",
}

r = httpx.post(
    f"http://127.0.0.1:{PORT}/v1/embeddings",
    headers={"Authorization": f"Bearer {API_KEY}"},
    json=payload,
    timeout=60.0,
)
r.raise_for_status()
body = r.json()

vectors = np.asarray([item["embedding"] for item in body["data"]], dtype=np.float32)
print("shape:", vectors.shape)
print("L2 norms:", np.linalg.norm(vectors, axis=1))
assert vectors.shape == (2, 2560)
assert np.allclose(np.linalg.norm(vectors, axis=1), 1.0, atol=1e-3)



## 6) Milvus/Zilliz direct ASR search

Collection phải chứa **Qwen3-Embedding-4B full 2560-dim, L2-normalized document vectors**. Schema server đang kỳ vọng phù hợp với `MilvusTextEmbeddingSink` trong repo:

```text
id
vector
keyframe_id
video_id
text
doc_type
model_version
metadata_json
```

Và index dùng `COSINE`.

Nếu ASR embedding artifact của bạn được import bằng tên collection khác, đặt `MILVUS_COLLECTION` **trước khi launch server** rồi chạy lại cell launch.

Direct endpoint:

```http
POST /v1/asr/search
Authorization: Bearer <SERVER_API_KEY>

{
  "query": "người dẫn chương trình nói về giá xăng",
  "top_k": 50,
  "min_score": 0.35
}
```


In [ ]:

# 6) Optional direct Milvus search test.

import os
import httpx

if not os.environ.get("MILVUS_URI", "").strip():
    print("MILVUS_URI is empty -> skipping direct Milvus test.")
else:
    PORT = int(os.environ.get("PORT", "8004"))
    API_KEY = os.environ["SERVER_API_KEY"]

    r = httpx.post(
        f"http://127.0.0.1:{PORT}/v1/asr/search",
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={
            "query": "người dẫn chương trình đang nói chuyện về kinh tế",
            "top_k": 20,
        },
        timeout=60.0,
    )
    r.raise_for_status()
    result = r.json()
    print("collection:", result["collection"])
    print("hits:", result["count"])
    for hit in result["hits"][:5]:
        print(hit["score"], hit.get("video_id"), hit.get("text"))



## 7) Connect với backend hiện tại của repo

Repo đã có `OpenAICompatibleTextEmbedder`, payload đúng dạng:

```json
{
  "model": "Qwen/Qwen3-Embedding-4B",
  "input": "user query"
}
```

nên endpoint `/v1/embeddings` của notebook tương thích.

### Khuyến nghị flow

```text
React
  -> apps/backend
      -> Qwen Kaggle /v1/embeddings
      -> Milvus ASR collection
      -> fuse ASR score với visual / OCR / caption
  -> React ranked results
```

Không nên đặt `SERVER_API_KEY` trong frontend React. Backend giữ key và gọi Kaggle server.

### Registry entry gợi ý

Backend hiện build OpenAI-compatible embedding adapters từ group `embedders`. Nếu bạn muốn register Qwen server trong cùng cơ chế adapter hiện tại, entry có thể bắt đầu như sau:

```yaml
embedders:
  qwen3_asr_embedding_4b:
    task: text_embedding
    provider: openai_compatible
    base_url: https://YOUR_PUBLIC_TUNNEL/v1
    model: Qwen/Qwen3-Embedding-4B
    api_key_env: QWEN3_ASR_SERVER_API_KEY
    dim: 2560
    l2_normalize: true
    collection: asr_embeddings_qwen3_embedding_4b_v1
    enabled: true
```

**Lưu ý:** retrieval service hiện dùng `embedders` chủ yếu cho semantic collection flow của frame embeddings. Để fuse ASR-vector score như một text source riêng biệt, bạn nên thêm một ASR-vector retrieval branch trong backend thay vì thay thế CLIP/SigLIP embedder mặc định. Endpoint `/v1/asr/search` trong notebook cho phép triển khai branch đó rất nhanh.


In [ ]:

# 8) Optional ngrok public tunnel.
# Recommended: create Kaggle Secret named NGROK_AUTHTOKEN.
# The URL is ephemeral unless your ngrok plan/domain is configured otherwise.

import os

token = os.environ.get("NGROK_AUTHTOKEN", "").strip()

if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
    except Exception:
        token = ""

if not token:
    print("No NGROK_AUTHTOKEN found. Skip tunnel.")
else:
    from pyngrok import ngrok

    ngrok.set_auth_token(token)
    PORT = int(os.environ.get("PORT", "8004"))
    tunnel = ngrok.connect(PORT, "http")
    PUBLIC_URL = tunnel.public_url.replace("http://", "https://")
    print("Public base URL:", PUBLIC_URL)
    print("Embedding base URL for backend:", PUBLIC_URL + "/v1")
    print("ASR direct search URL:", PUBLIC_URL + "/v1/asr/search")



## 9) Load benchmark

Benchmark này bắn nhiều request song song vào `/v1/embeddings` để đo:

- requests/second
- p50 latency
- p95 latency
- p99 latency

Dynamic batching sẽ có lợi nhất khi nhiều request ngắn đến gần nhau. Sau benchmark, xem `/metrics` và `nvidia-smi` để điều chỉnh:

- `INFER_BATCH_SIZE`: bắt đầu `8`; thử `12` hoặc `16` nếu VRAM còn nhiều.
- `MAX_BATCH_TEXTS`: `32` phù hợp high concurrency; có thể thử `48/64`.
- `BATCH_WAIT_MS`: `5–10 ms` thường là vùng hợp lý giữa latency và throughput.
- `QUEUE_MAX_REQUESTS`: backpressure; không tăng vô hạn.
- **Giữ `--workers 1`** cho Uvicorn.

Đừng chốt RPS bằng ước lượng. Hãy lấy số benchmark thực tế trên đúng Kaggle session vì CPU, network, query length và runtime version đều ảnh hưởng.


In [ ]:

# 9) Concurrent local benchmark.

import asyncio
import os
import statistics
import time
import httpx

PORT = int(os.environ.get("PORT", "8004"))
API_KEY = os.environ["SERVER_API_KEY"]
URL = f"http://127.0.0.1:{PORT}/v1/embeddings"

QUERIES = [
    "người đàn ông đang nói về kinh tế",
    "phỏng vấn vận động viên sau cuộc thi",
    "người phụ nữ nói về món ăn Việt Nam",
    "bản tin thời sự nói về giao thông",
    "cuộc trò chuyện về giáo dục đại học",
    "người dẫn chương trình nói về công nghệ",
    "phát biểu tại một sự kiện từ thiện",
    "bài phỏng vấn nói về bóng đá",
]

async def run_benchmark(total_requests=100, concurrency=32):
    semaphore = asyncio.Semaphore(concurrency)
    latencies = []
    failures = 0

    limits = httpx.Limits(
        max_connections=concurrency,
        max_keepalive_connections=concurrency,
    )

    async with httpx.AsyncClient(
        limits=limits,
        timeout=60.0,
        headers={"Authorization": f"Bearer {API_KEY}"},
    ) as client:

        async def one(i):
            nonlocal failures
            async with semaphore:
                started = time.perf_counter()
                try:
                    r = await client.post(
                        URL,
                        json={
                            "model": "Qwen/Qwen3-Embedding-4B",
                            "input": QUERIES[i % len(QUERIES)],
                            "input_type": "query",
                        },
                    )
                    r.raise_for_status()
                except Exception as exc:
                    failures += 1
                    print("failure:", repr(exc))
                finally:
                    latencies.append((time.perf_counter() - started) * 1000.0)

        started = time.perf_counter()
        await asyncio.gather(*(one(i) for i in range(total_requests)))
        elapsed = time.perf_counter() - started

    latencies_sorted = sorted(latencies)
    def percentile(p):
        if not latencies_sorted:
            return float("nan")
        idx = min(len(latencies_sorted) - 1, int(round((p / 100) * (len(latencies_sorted) - 1))))
        return latencies_sorted[idx]

    return {
        "requests": total_requests,
        "concurrency": concurrency,
        "failures": failures,
        "elapsed_s": round(elapsed, 3),
        "rps": round(total_requests / elapsed, 3),
        "p50_ms": round(percentile(50), 2),
        "p95_ms": round(percentile(95), 2),
        "p99_ms": round(percentile(99), 2),
    }

result = await run_benchmark(total_requests=100, concurrency=32)
result


In [ ]:

# 10) Inspect server batching metrics and GPU utilization snapshot.

import os
import subprocess
import httpx

PORT = int(os.environ.get("PORT", "8004"))
API_KEY = os.environ["SERVER_API_KEY"]

metrics = httpx.get(
    f"http://127.0.0.1:{PORT}/metrics",
    headers={"Authorization": f"Bearer {API_KEY}"},
    timeout=10.0,
).json()

print(metrics)
print()
print(
    subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
            "--format=csv",
        ],
        capture_output=True,
        text=True,
    ).stdout
)



## Tuning rule cho Kaggle T4×2

Thiết kế mặc định ưu tiên **throughput ổn định + không OOM**, không cố đoán một con số RPS cố định.

1. Giữ **2 model replicas**, mỗi T4 một replica.
2. T4 dùng **FP16**, không BF16.
3. Không tensor-parallel model 4B qua 2 T4 cho workload query embedding ngắn; data-parallel replicas thường hợp lý hơn để tăng request throughput.
4. Dùng bounded queue + HTTP 429 khi quá tải thay vì cho RAM tăng vô hạn.
5. Dynamic batching gom request trong khoảng vài ms để tăng GPU occupancy.
6. CPU được chia theo số GPU workers cho tokenizer/thread pools.
7. Milvus có client pool riêng để vector DB I/O không khóa event loop.
8. Query vector luôn 2560-dim + L2 normalized để khớp ASR embedding artifact hiện tại.
9. Query dùng retrieval instruction; ASR documents vẫn giữ raw text khi offline embedding.
10. Kaggle notebook là ephemeral compute. Cấu hình này tối ưu trong giới hạn session Kaggle, nhưng không thay thế hạ tầng production có autoscaling/SLA.


In [ ]:

# 11) Stop server when you are done.

import os
import signal
from pathlib import Path

PID_FILE = Path("/kaggle/working/qwen3_server.pid")
if PID_FILE.exists():
    pid = int(PID_FILE.read_text().strip())
    try:
        os.kill(pid, signal.SIGTERM)
        print("Sent SIGTERM to", pid)
    except ProcessLookupError:
        print("Server process is already gone.")
    PID_FILE.unlink(missing_ok=True)
else:
    print("No PID file found.")
